# 107 — Knowledge graphs y GraphRAG

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Knowledge graph**: conocimiento como tripletas **(sujeto, predicado, objeto)** —
nodos = entidades, aristas = relaciones tipadas y dirigidas; en grafos de propiedades,
con atributos (`año`, `fuente`) en nodos y aristas. Aporta composicionalidad (multi-hop
= recorrer caminos), agregación (consultas, no lecturas) y procedencia por hecho.

**Cypher** (conceptual): patrones de camino declarativos —
`MATCH (p:Persona)-[:FUNDÓ]->(e)-[:ADQUIRIÓ]->(x {nombre:"Instagram"}) RETURN p.nombre`.

**Extracción con LLM**: lo difícil no es extraer sino la **resolución de entidades**
("IBM" = "International Business Machines" = "la compañía") y la **normalización de
relaciones** (fundó/creó/estableció → un solo predicado).

**GraphRAG** (arXiv:2404.16130): para preguntas **globales** que ningún top-k local
cubre. Indexación: extraer grafo → comunidades jerárquicas (Leiden) → resumen LLM por
comunidad. Consulta global = map-reduce sobre resúmenes; consulta local = vecindario de
las entidades mencionadas + chunks fuente.

## 🧮 Ejemplo de referencia

Del párrafo "Marie Curie descubrió el polonio en 1898 junto a su esposo Pierre. Por sus
investigaciones sobre la radiactividad recibió el Nobel de Física en 1903, compartido
con Pierre y Henri Becquerel":

```text
(Marie Curie,  DESCUBRIÓ,  polonio)          {año: 1898}
(Pierre Curie, DESCUBRIÓ,  polonio)          {año: 1898}
(Marie Curie,  CASADA_CON, Pierre Curie)
(Marie Curie,  RECIBIÓ,    Nobel de Física)  {año: 1903}
(Pierre Curie, RECIBIÓ,    Nobel de Física)  {año: 1903}
(H. Becquerel, RECIBIÓ,    Nobel de Física)  {año: 1903}
```

"su esposo Pierre" exige resolver la correferencia; "compartido con" genera tres
tripletas RECIBIÓ; el año es propiedad de la arista, no un nodo.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("logic", seed=107)
show(result)


## Reflexión

1. ¿Por qué la pregunta "¿cuáles son los tres temas dominantes de este corpus?" es inalcanzable para un RAG vectorial top-k, y qué componente de GraphRAG la hace posible?
2. Si la resolución de entidades falla y "Meta" y "Facebook Inc." quedan como nodos separados, ¿qué tipo de consultas devuelven resultados incompletos sin dar ningún error?
3. El grafo extraído por LLM tiene apariencia de base de datos pero hereda alucinaciones del modelo. ¿Qué proceso de verificación propondrías antes de tratar una tripleta como hecho?